# Clustering con *k*-means

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

Se genera una base de datos con puntos ubicados aleatoriamente dentro de un número dado de grupos.

In [ ]:
n_points = 10000
print("Cantidad de puntos:", n_points)

In [ ]:
# fijar la semilla para obtener resultados reproducibles
np.random.seed(0)

In [ ]:
from sklearn.datasets import make_blobs
n_clusters = 10
dataset, _ = make_blobs(n_samples=n_points, n_features=2, centers=n_clusters)
print("Se generó una base de datos con", n_clusters, "grupos en", dataset.shape[1], "dimensiones.")

In [ ]:
plt.plot(dataset[:,0], dataset[:,1], 'b.')
plt.title('Base de datos')
plt.show()

El algoritmo de *k*-means es un método iterativo que busca *k* conglomeraciones. Cada conglomeración está representada por un centro, y cada punto se asigna al centro más cercano en el espacio.

Como todo método iterativo, requiere una aproximación inicial. Aquí, los centros iniciales están distribuidos uniformemente en el segmento $0<x<1$, $y=0$.

In [ ]:
k = 5
print("Cantidad de conglomeraciones:", k)

In [ ]:
centers = np.zeros([k, 2])
for c in range(k):
    centers[c,0] = float(c+1)/(k+1)
    centers[c,1] = 0.0

In [ ]:
plt.plot(centers[:,0], centers[:,1], 'rx')
plt.xlim([0,1])
plt.ylim([-1,1])
plt.title('Centros iniciales')
plt.show()

El algoritmo de *k*-means mejora iterativamente la ubicación de los centros mediante dos pasos.

 1. **Asignación:** se recorren todos los puntos y se busca el centro más cercano a cada uno. Cada punto recibe la etiqueta del centro más cercano, es decir, de su conglomeración.

 1. **Actualización:** se calcula el centro de masa de cada conglomeración, que pasa a ser su nuevo centro.

Estos dos pasos se repiten hasta que las conglomeraciones ya no cambien, o hasta alcanzar el número máximo de iteraciones.

In [ ]:
max_iterations = 100

In [ ]:
%%time

# guardar el historial de centros y conglomeraciones
history_of_centers = [centers]
history_of_clusters = []

for iteration in range(max_iterations):
    
    old_centers = history_of_centers[-1]

    # paso de asignación
    cluster = np.zeros(n_points, dtype='int')
    for p in range(n_points):
        minimum_distance = np.inf
        nearest_cluster = None
        for c in range(k):
            distance_to_cluster_c = np.sqrt((dataset[p,0] - old_centers[c,0])**2 + (dataset[p,1] - old_centers[c,1])**2)
            if distance_to_cluster_c < minimum_distance:
                minimum_distance = distance_to_cluster_c
                nearest_cluster = c
        cluster[p] = nearest_cluster
    history_of_clusters.append(cluster)
    
    # revisar convergencia
    if iteration > 0:
        clusters_changed = False
        for p in range(n_points):
            if history_of_clusters[-1][p] != history_of_clusters[-2][p]:
                clusters_changed = True
        if not clusters_changed:
            print("Convergió en", iteration, "iteraciones.\n")
            break

    # paso de actualización
    number_of_points_in_cluster = np.zeros(k, dtype='int')
    sum_of_locations = np.zeros([k, 2], dtype='float')
    for p in range(n_points):
        number_of_points_in_cluster[cluster[p]] += 1
        sum_of_locations[cluster[p], :] += dataset[p, :]
    new_centers = np.zeros([k, 2], dtype='float')
    for c in range(k):
        new_centers[c,0] = sum_of_locations[c,0] / number_of_points_in_cluster[c]
        new_centers[c,1] = sum_of_locations[c,1] / number_of_points_in_cluster[c]
    history_of_centers.append(new_centers)

if iteration == max_iterations-1:
    print("El algoritmo no convergió en", max_iterations, "iteraciones.")

Se visualizan las conglomeraciones en cada iteración.

In [ ]:
for itr in range(iteration+1):
    plt.scatter(dataset[:,0], dataset[:,1], c=history_of_clusters[itr])
    for l in range(itr):
        plt.plot(history_of_centers[l][:,0], history_of_centers[l][:,1], 'b+')
    plt.plot(history_of_centers[itr][:,0], history_of_centers[itr][:,1], 'rx')
    plt.title("Iteración "+str(itr))
    plt.show()

Efectivamente, el algoritmo encuentra conglomeraciones de puntos.

La inercia es la suma de las distancias al cuadrado de cada punto al centro de su conglomeración:
$$I_k = \sum_{i=1}^k \sum_{j=1}^{N_i} \|\mathbf{x}_{ij} - \mathbf{c}_i\|^2.$$

In [ ]:
def calc_inertia(datapoints, cluster, centers):
    inertia = 0.0
    for i in range(centers.shape[0]):
        inertia += np.sum( np.linalg.norm( datapoints[cluster==i, :] - centers[i,:], axis=1 )**2 )
    return inertia

In [ ]:
inertia = calc_inertia(dataset, history_of_clusters[-1], history_of_centers[-1])
print("La inercia de las conglomeraciones es", inertia)